# Bronze to Silver load (Arena and Epoch)

Reads the four-text Bronze tables, cleans, casts, validates, de-duplicates and writes **typed Silver tables with `MERGE INTO`**.
Same widget names as the Bronze notebook. Every table and batch writes a log row (layer `Bronze-to-Silver`).

**Normal run:** Run all with the default widgets. It processes every Bronze batch that Silver has not logged yet.
**Backfill:** `from_batch` / `to_batch` (or `batch_id`, `start_date`, `end_date`) plus `force_reload=true`.
**Rebuild from the first batch:** `replay=true`.
**Not in this version:** BenchLM and CooperBench (added after profiling is finished).

### CELL 1 - Parameters and configuration

In [ ]:
# CELL 1 - Parameters and configuration
import json, re, traceback, uuid
from datetime import date, datetime, timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import (BooleanType, DoubleType, LongType, StringType, StructField, StructType,
                               TimestampType)

dbutils.widgets.text("catalog", "workspace", "catalog")
dbutils.widgets.text("bronze_schema", "bronze", "bronze_schema")
dbutils.widgets.text("silver_schema", "silver", "silver_schema")
dbutils.widgets.dropdown("source", "all", ["all", "arena", "epoch"], "source (benchlm and cooperbench come later)")
dbutils.widgets.text("batch_id", "", "batch_id (one batch; empty = every pending batch)")
dbutils.widgets.text("from_batch", "", "from_batch (batch id or prefix, e.g. 20261005)")
dbutils.widgets.text("to_batch", "", "to_batch (batch id or prefix, inclusive)")
dbutils.widgets.text("start_date", "", "start_date YYYY-MM-DD (collection day)")
dbutils.widgets.text("end_date", "", "end_date YYYY-MM-DD (collection day)")
dbutils.widgets.dropdown("force_reload", "false", ["false", "true"], "force_reload: also process batches Silver already logged")
dbutils.widgets.dropdown("replay", "false", ["false", "true"], "replay: all batches, oldest first, one at a time")

P = {k: dbutils.widgets.get(k).strip() for k in
     ["catalog", "bronze_schema", "silver_schema", "source", "batch_id", "from_batch", "to_batch",
      "start_date", "end_date", "force_reload", "replay"]}

PARAM_ERRORS = []


def parse_day(name):
    value = P[name]
    if not value:
        return ""
    try:
        return date.fromisoformat(value).isoformat()
    except ValueError:
        PARAM_ERRORS.append(f"{name}='{value}' is not a valid YYYY-MM-DD date")
        return ""


START, END = parse_day("start_date"), parse_day("end_date")
if START and END and START > END:
    PARAM_ERRORS.append(f"start_date {START} is after end_date {END}")
FROM_B, TO_B = P["from_batch"], P["to_batch"]
if FROM_B and TO_B:
    _n = min(len(FROM_B), len(TO_B))
    if FROM_B[:_n] > TO_B[:_n]:
        PARAM_ERRORS.append(f"from_batch {FROM_B} is later than to_batch {TO_B}")

REPLAY = P["replay"] == "true"
FORCE = P["force_reload"] == "true" or REPLAY

CATALOG, BSCHEMA, SSCHEMA = P["catalog"], P["bronze_schema"], P["silver_schema"]
for _name in (CATALOG, BSCHEMA, SSCHEMA):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", _name):
        raise ValueError(f"Invalid catalog/schema name: {_name!r}")
SOURCES = ["arena", "epoch"] if P["source"] == "all" else [P["source"]]

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:6]
spark.conf.set("spark.sql.session.timeZone", "UTC")

T_RUNS = f"{CATALOG}.ops.pipeline_execution_logs"
T_ERRORS = f"{CATALOG}.ops.error_logs"
T_DRIFT = f"{CATALOG}.ops.schema_drift_log"
T_SQ = f"{CATALOG}.ops.silver_quarantine_log"
LAYER = "Bronze-to-Silver"

print("Run id:", RUN_ID, "| sources:", SOURCES, "| force:", FORCE, "| parameters:", P)

### CELL 2 - Explicit schemas (the Silver data contracts). Casting is explicit in these column maps; nothing is inferred.

In [ ]:
# CELL 2 - Explicit schemas (the Silver data contracts)

def q(name):
    return "`" + name.replace("`", "``") + "`"


# ---- Arena: Bronze text columns that Silver reads -> silver.arena_leaderboard ----
ARENA_RAW_COLS = ["subset", "category", "model_name", "organization", "license", "leaderboard_publish_date",
                  "rating", "variance", "rating_q975", "rating_q025", "rating_lower", "rating_upper",
                  "score", "score_ci_lower", "score_ci_upper", "vote_count", "observation_count",
                  "session_count", "rank"]
ARENA_SUBSETS = ["text", "text_style_control", "webdev", "agent"]
ARENA_SILVER = [("subset", "STRING"), ("category", "STRING"), ("model_name", "STRING"), ("model_key", "STRING"),
                ("organization", "STRING"), ("license", "STRING"), ("leaderboard_publish_date", "DATE"),
                ("rating", "DOUBLE"), ("rating_variance", "DOUBLE"), ("rating_ci_lower", "DOUBLE"),
                ("rating_ci_upper", "DOUBLE"), ("vote_count", "BIGINT"), ("observation_count", "BIGINT"),
                ("session_count", "BIGINT"), ("rank", "BIGINT"), ("bronze_record_hash", "STRING"),
                ("bronze_batch_id", "STRING"), ("source_file", "STRING"), ("collection_timestamp", "TIMESTAMP"),
                ("load_timestamp", "TIMESTAMP")]
ARENA_KEYS = ["subset", "category", "model_name", "leaderboard_publish_date"]

# ---- Epoch: (silver column, Bronze column as published, type) -> silver.epoch_models ----
EPOCH_MAP = [
    ("model", "Model", "str"), ("domain", "Domain", "str"), ("task", "Task", "str"),
    ("organization", "Organization", "str"), ("authors", "Authors", "str"),
    ("publication_date", "Publication date", "date"), ("reference", "Reference", "str"),
    ("link", "Link", "str"), ("citations", "Citations", "big"),
    ("notability_criteria", "Notability criteria", "str"),
    ("notability_criteria_notes", "Notability criteria notes", "str"),
    ("parameters", "Parameters", "dbl"), ("training_compute_flop", "Training compute (FLOP)", "dbl"),
    ("training_dataset_size_total", "Training dataset size (total)", "dbl"),
    ("training_time_hours", "Training time (hours)", "dbl"), ("training_hardware", "Training hardware", "str"),
    ("approach", "Approach", "str"), ("confidence", "Confidence", "str"), ("epochs", "Epochs", "dbl"),
    ("model_accessibility", "Model accessibility", "str"),
    ("country_of_organization", "Country (of organization)", "str"), ("base_model", "Base model", "str"),
    ("finetune_compute_flop", "Finetune compute (FLOP)", "dbl"),
    ("hardware_quantity", "Hardware quantity", "dbl"),
    ("hardware_utilization_mfu", "Hardware utilization (MFU)", "dbl"),
    ("last_modified", "Last modified", "ts"),
    ("organization_categorization", "Organization categorization", "str"),
    ("foundation_model", "Foundation model", "str"), ("frontier_model", "Frontier model", "str"),
    ("training_compute_cost_usd_2023", "Training compute cost (2023 USD)", "dbl"),
    ("training_power_draw_w", "Training power draw (W)", "dbl"),
    ("open_model_weights", "Open model weights?", "bool"),
    ("hugging_face_developer_id", "Hugging Face developer id", "str"),
]
EPOCH_NONNEG = ["citations", "parameters", "training_compute_flop", "training_dataset_size_total",
                "training_time_hours", "epochs", "finetune_compute_flop", "hardware_quantity",
                "training_compute_cost_usd_2023", "training_power_draw_w"]
_SQL_TYPE = {"str": "STRING", "dbl": "DOUBLE", "big": "BIGINT", "date": "DATE", "ts": "TIMESTAMP", "bool": "BOOLEAN"}
EPOCH_SILVER = ([("model", "STRING"), ("model_key", "STRING")]
                + [(s, _SQL_TYPE[t]) for s, b, t in EPOCH_MAP if s != "model"]
                + [("snapshot_id", "STRING"), ("payload_hash", "STRING"), ("bronze_record_hash", "STRING"),
                   ("bronze_batch_id", "STRING"), ("source_file", "STRING"),
                   ("collection_timestamp", "TIMESTAMP"), ("load_timestamp", "TIMESTAMP")])
EPOCH_KEYS = ["model", "publication_date"]

LINEAGE_RAW = ["record_hash", "batch_id", "source_file", "collection_timestamp", "raw_payload"]
BRONZE_META = {"load_type", "raw_payload", "record_hash", "source_file", "batch_id", "collection_timestamp",
               "load_timestamp", "snapshot_id", "payload_hash"}

# ---- operations tables (same layout as the Bronze notebook; the new one is silver_quarantine_log) ----
OPS_TYPES = {"string": StringType(), "bigint": LongType(), "double": DoubleType(), "boolean": BooleanType(),
             "timestamp": TimestampType()}
OPS_SPECS = {
    T_RUNS: [("run_id", "string"), ("batch_id", "string"), ("layer", "string"), ("source", "string"),
             ("dataset", "string"), ("load_type", "string"), ("file_or_param", "string"),
             ("parameters_json", "string"), ("start_time", "timestamp"), ("end_time", "timestamp"),
             ("duration_seconds", "double"), ("status", "string"), ("rows_read", "bigint"),
             ("rows_inserted", "bigint"), ("rows_updated", "bigint"), ("rows_skipped_duplicate", "bigint"),
             ("rows_quarantined", "bigint"), ("bytes_processed", "bigint"), ("message", "string"),
             ("load_timestamp", "timestamp")],
    T_ERRORS: [("error_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("component", "string"),
               ("layer", "string"), ("source", "string"), ("file_or_param", "string"), ("severity", "string"),
               ("error_type", "string"), ("error_message", "string"), ("http_status", "bigint"),
               ("attempt", "bigint"), ("stack_trace", "string"), ("occurred_at", "timestamp"),
               ("resolved", "boolean"), ("load_timestamp", "timestamp")],
    T_DRIFT: [("drift_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("dataset", "string"),
              ("source_file", "string"), ("column_name", "string"), ("event_type", "string"),
              ("expected_type", "string"), ("actual_type", "string"), ("action_taken", "string"),
              ("detected_at", "timestamp"), ("load_timestamp", "timestamp")],
    T_SQ: [("quarantine_id", "string"), ("run_id", "string"), ("batch_id", "string"), ("silver_table", "string"),
           ("bronze_record_hash", "string"), ("column_name", "string"), ("bad_value", "string"),
           ("reason", "string"), ("raw_payload", "string"), ("quarantined_at", "timestamp"),
           ("load_timestamp", "timestamp")],
}

### CELL 3 - Create the Silver tables and the Silver quarantine log (every table has load_timestamp)

In [ ]:
# CELL 3 - Create the Silver tables and the Silver quarantine log (every table has load_timestamp)

T_S_ARENA = f"{CATALOG}.{SSCHEMA}.arena_leaderboard"
T_S_EPOCH = f"{CATALOG}.{SSCHEMA}.epoch_models"
T_B_ARENA = f"{CATALOG}.{BSCHEMA}.arena_leaderboard_raw"
T_B_EPOCH = f"{CATALOG}.{BSCHEMA}.epoch_models_raw"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.ops")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SSCHEMA}")
for _table, _spec in OPS_SPECS.items():
    spark.sql(f"CREATE TABLE IF NOT EXISTS {_table} ({', '.join(q(n) + ' ' + t.upper() for n, t in _spec)}) USING DELTA")
for _table, _spec in [(T_S_ARENA, ARENA_SILVER), (T_S_EPOCH, EPOCH_SILVER)]:
    spark.sql(f"CREATE TABLE IF NOT EXISTS {_table} ({', '.join(q(n) + ' ' + t for n, t in _spec)}) USING DELTA")
print("Tables ready:", T_S_ARENA, T_S_EPOCH, T_SQ)

### CELL 4 - Logging helpers (execution log, error log, drift log, quarantine log)

In [ ]:
# CELL 4 - Logging helpers


def now_iso():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S.%fZ")


def append_record(table, record):
    spec = OPS_SPECS[table]
    schema = StructType([StructField(n, StringType() if t == "timestamp" else OPS_TYPES[t], True) for n, t in spec])
    df = spark.createDataFrame([tuple(record.get(n) for n, _ in spec)], schema)
    for name, kind in spec:
        if kind == "timestamp":
            df = df.withColumn(name, F.to_timestamp(F.col(name)))
    df.write.mode("append").saveAsTable(table)


class Counts:
    def __init__(self):
        self.read = self.inserted = self.updated = self.skipped = self.quarantined = 0

    def add(self, other):
        for k in ("read", "inserted", "updated", "skipped", "quarantined"):
            setattr(self, k, getattr(self, k) + getattr(other, k))


def log_run(layer, batch_id, file_or_param, source, dataset, start, status, c=None, message="", params=None):
    end = now_iso()
    c = c or Counts()
    duration = (datetime.strptime(end, "%Y-%m-%dT%H:%M:%S.%fZ")
                - datetime.strptime(start, "%Y-%m-%dT%H:%M:%S.%fZ")).total_seconds()
    append_record(T_RUNS, {
        "run_id": RUN_ID, "batch_id": batch_id, "layer": layer, "source": source, "dataset": dataset,
        "load_type": None, "file_or_param": file_or_param,
        "parameters_json": json.dumps(params if params is not None else P), "start_time": start, "end_time": end,
        "duration_seconds": duration, "status": status, "rows_read": c.read, "rows_inserted": c.inserted,
        "rows_updated": c.updated, "rows_skipped_duplicate": c.skipped, "rows_quarantined": c.quarantined,
        "bytes_processed": None, "message": str(message)[:4000], "load_timestamp": now_iso()})


def log_error(source, file_or_param, message, error_type="", batch_id="", stack="", severity="ERROR"):
    print(f"{severity} | {source} | {file_or_param} | {message}")
    try:
        append_record(T_ERRORS, {
            "error_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id, "component": "silver_notebook",
            "layer": LAYER, "source": source, "file_or_param": file_or_param, "severity": severity,
            "error_type": error_type, "error_message": str(message)[:4000], "http_status": None, "attempt": None,
            "stack_trace": stack[-4000:], "occurred_at": now_iso(), "resolved": False, "load_timestamp": now_iso()})
    except Exception as exc:
        print("Could not write to the error table:", exc)


KNOWN_DRIFT = set()   # (dataset, column, event) already logged by an earlier run


def load_known_drift():
    rows = (spark.table(T_DRIFT).filter(F.col("dataset").startswith("silver/"))
            .select("dataset", "column_name", "event_type").distinct().collect())
    return {(r["dataset"], r["column_name"], r["event_type"]) for r in rows}


def log_drift(batch_id, dataset, source_file, column, event, expected, actual, action, once=True):
    if once and (dataset, column, event) in KNOWN_DRIFT:
        return
    KNOWN_DRIFT.add((dataset, column, event))
    append_record(T_DRIFT, {"drift_id": uuid.uuid4().hex, "run_id": RUN_ID, "batch_id": batch_id,
                            "dataset": dataset, "source_file": source_file, "column_name": column,
                            "event_type": event, "expected_type": expected, "actual_type": actual,
                            "action_taken": action, "detected_at": now_iso(), "load_timestamp": now_iso()})


def quarantine_df_to_table(df, silver_label, batch_id):
    """Insert-only MERGE on a deterministic id, so repeating a run never adds the same bad record twice."""
    out = (df.select(
        F.sha2(F.concat_ws("|", F.lit(silver_label), F.col("bronze_record_hash"), F.col("column_name"),
                           F.col("reason")), 256).alias("quarantine_id"),
        F.lit(RUN_ID).alias("run_id"), F.lit(batch_id).alias("batch_id"), F.lit(silver_label).alias("silver_table"),
        F.col("bronze_record_hash"), F.col("column_name"), F.substring("bad_value", 1, 2000).alias("bad_value"),
        F.col("reason"), F.substring("raw_payload", 1, 4000).alias("raw_payload"),
        F.current_timestamp().alias("quarantined_at"), F.current_timestamp().alias("load_timestamp"))
        .dropDuplicates(["quarantine_id"]))
    view = "sq_" + uuid.uuid4().hex[:10]
    out.createOrReplaceTempView(view)
    cols = [n for n, _ in OPS_SPECS[T_SQ]]
    result = spark.sql(f"""MERGE INTO {T_SQ} AS t USING {view} AS s ON t.quarantine_id = s.quarantine_id
        WHEN NOT MATCHED THEN INSERT ({', '.join(q(c) for c in cols)}) VALUES ({', '.join('s.' + q(c) for c in cols)})"""
                       ).collect()[0].asDict()
    spark.catalog.dropTempView(view)
    return int(result.get("num_inserted_rows") or 0)

### CELL 5 - Cleaning, casting and validation (pure functions: Bronze text in, typed rows plus an issues list out)

In [ ]:
# CELL 5 - Cleaning, casting and validation (spec section 6)

NULL_MARKERS = ["", "nan", "null", "none", "n/a", "-"]


def clean(col):
    """Trim, collapse repeated spaces, turn empty strings and null markers into NULL."""
    t = F.trim(F.regexp_replace(col.cast("string"), r"\s+", " "))
    return F.when(F.lower(t).isin(*NULL_MARKERS), F.lit(None).cast("string")).otherwise(t)


def _dbl(n):
    return f"nullif(nullif(nullif(try_cast({q(n)} AS DOUBLE), double('NaN')), double('Infinity')), double('-Infinity'))"


def to_double(n):
    return F.expr(_dbl(n))


def to_bigint(n):
    d = _dbl(n)
    return F.expr(f"CASE WHEN abs({d}) < 9.0E18 AND {d} = floor({d}) THEN CAST({d} AS BIGINT) END")


def to_date(n):
    return F.expr(f"CAST(try_to_timestamp({q(n)}, 'yyyy-MM-dd') AS DATE)")


def to_ts(n):
    return F.expr(f"try_cast({q(n)} AS TIMESTAMP)")


def to_bool(n):
    return F.expr(f"CASE lower({q(n)}) WHEN 'true' THEN true WHEN 'yes' THEN true "
                  f"WHEN 'false' THEN false WHEN 'no' THEN false END")


CASTERS = {"str": lambda n: F.col(n), "dbl": to_double, "big": to_bigint, "date": to_date, "ts": to_ts,
           "bool": to_bool}


def model_key(col):
    return F.regexp_replace(F.regexp_replace(F.lower(col), "[^a-z0-9]+", "-"), "^-+|-+$", "")


def issue(cond, column, reason, bad_value):
    return F.when(cond, F.struct(F.lit(column).alias("column_name"), F.lit(reason).alias("reason"),
                                 bad_value.cast("string").alias("bad_value")))


def issues_array(items):
    return F.filter(F.array(*items), lambda x: x.isNotNull())


def neg(c):
    return F.coalesce(F.col(c) < 0, F.lit(False))


def nullify(df, flag, cols):
    for c in cols:
        df = df.withColumn(c, F.when(F.col(flag), F.lit(None)).otherwise(F.col(c)))
    return df


# ------------------------------------------------------------------ Arena
ARENA_CASTS = [(c, "c_" + c, to_double) for c in
               ["rating", "variance", "rating_q975", "rating_q025", "rating_lower", "rating_upper", "score",
                "score_ci_lower", "score_ci_upper"]] + \
              [(c, "c_" + c, to_bigint) for c in ["vote_count", "observation_count", "session_count", "rank"]] + \
              [("leaderboard_publish_date", "c_leaderboard_publish_date", to_date)]
ARENA_FLAGS = ["v_rating_ci", "v_rating_variance", "v_vote_count", "v_observation_count", "v_session_count",
               "v_rank"]


def arena_build(raw):
    d = raw.select(*[F.col(c).cast("string").alias("r_" + c) for c in ARENA_RAW_COLS],
                   F.col("record_hash").alias("bronze_record_hash"), F.col("batch_id").alias("bronze_batch_id"),
                   "source_file", "collection_timestamp", "raw_payload")
    d = d.select("*", *[clean(F.col("r_" + c)).alias("c_" + c) for c in ARENA_RAW_COLS])
    d = d.select(
        "*",
        F.col("c_subset").alias("subset"), F.col("c_category").alias("category"),
        F.col("c_model_name").alias("model_name"), model_key(F.col("c_model_name")).alias("model_key"),
        F.col("c_organization").alias("organization"), F.col("c_license").alias("license"),
        to_date("c_leaderboard_publish_date").alias("leaderboard_publish_date"),
        F.coalesce(to_double("c_rating"), to_double("c_score")).alias("rating"),
        to_double("c_variance").alias("rating_variance"),
        F.coalesce(to_double("c_rating_q025"), to_double("c_rating_lower"),
                   to_double("c_score_ci_lower")).alias("rating_ci_lower"),
        F.coalesce(to_double("c_rating_q975"), to_double("c_rating_upper"),
                   to_double("c_score_ci_upper")).alias("rating_ci_upper"),
        to_bigint("c_vote_count").alias("vote_count"), to_bigint("c_observation_count").alias("observation_count"),
        to_bigint("c_session_count").alias("session_count"), to_bigint("c_rank").alias("rank"))
    lo, hi, r = F.col("rating_ci_lower"), F.col("rating_ci_upper"), F.col("rating")
    d = d.select(
        "*",
        ((lo.isNotNull() & r.isNotNull() & (lo > r)) | (hi.isNotNull() & r.isNotNull() & (hi < r))
         | (lo.isNotNull() & hi.isNotNull() & (lo > hi))).alias("v_rating_ci"),
        neg("rating_variance").alias("v_rating_variance"), neg("vote_count").alias("v_vote_count"),
        neg("observation_count").alias("v_observation_count"), neg("session_count").alias("v_session_count"),
        F.coalesce(F.col("rank") < 1, F.lit(False)).alias("v_rank"))
    d = d.withColumn("_issues", issues_array([
        issue(F.coalesce(~F.col("subset").isin(*ARENA_SUBSETS), F.lit(True)), "subset", "SUBSET_NOT_ALLOWED",
              F.col("r_subset")),
        issue(F.col("category").isNull(), "category", "CATEGORY_MISSING", F.col("r_category")),
        issue(F.col("model_name").isNull(), "model_name", "MODEL_NAME_MISSING", F.col("r_model_name")),
        issue(F.col("leaderboard_publish_date").isNull()
              | (F.col("leaderboard_publish_date") < F.lit("2023-01-01").cast("date"))
              | (F.col("leaderboard_publish_date") > F.date_add(F.current_date(), 1)),
              "leaderboard_publish_date", "PUBLISH_DATE_INVALID_OR_OUT_OF_RANGE",
              F.col("r_leaderboard_publish_date")),
        issue(F.col("rating").isNull() | (F.col("rating") <= 0), "rating", "RATING_MISSING_OR_NOT_POSITIVE",
              F.coalesce(F.col("r_rating"), F.col("r_score")))]))
    d = nullify(d, "v_rating_ci", ["rating_ci_lower", "rating_ci_upper"])
    for flag, col in [("v_rating_variance", "rating_variance"), ("v_vote_count", "vote_count"),
                      ("v_observation_count", "observation_count"), ("v_session_count", "session_count"),
                      ("v_rank", "rank")]:
        d = nullify(d, flag, [col])
    return d


# ------------------------------------------------------------------ Epoch
EPOCH_CASTS = [(s, "c_" + s, CASTERS[t]) for s, b, t in EPOCH_MAP if t != "str"]
EPOCH_FLAGS = ["v_" + s for s in EPOCH_NONNEG] + ["v_link"]


def epoch_build(raw):
    d = raw.select(*[F.col(q(b)).cast("string").alias("r_" + s) for s, b, t in EPOCH_MAP],
                   F.col("record_hash").alias("bronze_record_hash"), F.col("batch_id").alias("bronze_batch_id"),
                   "snapshot_id", "payload_hash", "source_file", "collection_timestamp", "raw_payload")
    d = d.select("*", *[clean(F.col("r_" + s)).alias("c_" + s) for s, b, t in EPOCH_MAP])
    d = d.select("*", *[CASTERS[t]("c_" + s).alias(s) for s, b, t in EPOCH_MAP],
                 model_key(F.col("c_model")).alias("model_key"))
    d = d.select("*", *[neg(s).alias("v_" + s) for s in EPOCH_NONNEG],
                 (F.col("link").isNotNull() & ~F.lower(F.col("link")).startswith("http")).alias("v_link"))
    d = d.withColumn("_issues", issues_array([
        issue(F.col("model").isNull(), "model", "MODEL_MISSING", F.col("r_model")),
        issue(F.col("publication_date").isNull()
              | (F.col("publication_date") < F.lit("1900-01-01").cast("date"))
              | (F.col("publication_date") > F.date_add(F.current_date(), 1)),
              "publication_date", "PUBLICATION_DATE_INVALID_OR_OUT_OF_RANGE", F.col("r_publication_date"))]))
    for s in EPOCH_NONNEG:
        d = nullify(d, "v_" + s, [s])
    d = nullify(d, "v_link", ["link"])
    return d


# ------------------------------------------------------------------ per-source configuration
CFG = {
    "arena": dict(
        name="arena", bronze=T_B_ARENA, silver=T_S_ARENA, bronze_short=f"{BSCHEMA}.arena_leaderboard_raw",
        silver_short=f"{SSCHEMA}.arena_leaderboard", spec=ARENA_SILVER, keys=ARENA_KEYS,
        raw_cols=ARENA_RAW_COLS, extra_raw=[], build=arena_build, casts=ARENA_CASTS, flags=ARENA_FLAGS,
        change_col="bronze_record_hash", chunk="subset", promoted=set(ARENA_RAW_COLS),
        order=lambda: [F.col("collection_timestamp").desc(), F.col("vote_count").desc_nulls_last(),
                       F.col("bronze_batch_id").desc(), F.col("bronze_record_hash")]),
    "epoch": dict(
        name="epoch", bronze=T_B_EPOCH, silver=T_S_EPOCH, bronze_short=f"{BSCHEMA}.epoch_models_raw",
        silver_short=f"{SSCHEMA}.epoch_models", spec=EPOCH_SILVER, keys=EPOCH_KEYS,
        raw_cols=[b for s, b, t in EPOCH_MAP], extra_raw=["snapshot_id", "payload_hash"], build=epoch_build,
        casts=EPOCH_CASTS, flags=EPOCH_FLAGS, change_col="payload_hash", chunk=None,
        promoted={b for s, b, t in EPOCH_MAP},
        order=lambda: [F.col("collection_timestamp").desc(), F.col("bronze_batch_id").desc(),
                       F.col("bronze_record_hash")]),
}


def prepare(cfg, typed):
    """Split typed rows into: bad records, the de-duplicated source for the MERGE, and duplicate losers."""
    out_cols = [F.col(n).cast(t.lower()).alias(n) for n, t in cfg["spec"] if n != "load_timestamp"]
    bad = typed.filter(F.size("_issues") > 0)
    valid = typed.filter(F.size("_issues") == 0)
    w = Window.partitionBy(*[F.col(k) for k in cfg["keys"]]).orderBy(*cfg["order"]())
    ranked = valid.withColumn("_rn", F.row_number().over(w))
    src = ranked.filter(F.col("_rn") == 1).select(*out_cols, F.current_timestamp().alias("load_timestamp"))
    losers = ranked.filter(F.col("_rn") > 1)
    return bad, src, losers


def quarantine_rows_bad(bad):
    ex = bad.select("bronze_record_hash", "raw_payload", F.explode("_issues").alias("i"))
    return ex.select("bronze_record_hash", "raw_payload", F.col("i.column_name").alias("column_name"),
                     F.col("i.reason").alias("reason"), F.col("i.bad_value").alias("bad_value"))


def quarantine_rows_losers(losers, keys):
    return losers.select("bronze_record_hash", "raw_payload", F.lit("business_key").alias("column_name"),
                         F.lit("DUPLICATE_BUSINESS_KEY").alias("reason"),
                         F.concat_ws("|", *[F.col(k).cast("string") for k in keys]).alias("bad_value"))

### CELL 6 - Engine: read one Bronze batch, clean, validate, de-duplicate, MERGE (one source, one batch)

In [ ]:
# CELL 6 - Engine

def ensure_silver_columns(cfg, batch_id):
    """Silver evolves only through the column map: a mapped column that the table lacks is added and logged."""
    have = {f.name.lower() for f in spark.table(cfg["silver"]).schema.fields}
    for name, typ in cfg["spec"]:
        if name.lower() not in have:
            spark.sql(f"ALTER TABLE {cfg['silver']} ADD COLUMNS ({q(name)} {typ})")
            log_drift(batch_id, f"silver/{cfg['name']}", cfg["bronze_short"], name, "NEW_COLUMN", "", typ,
                      "TABLE_EVOLVED")


def check_bronze_structure(cfg, batch_id):
    cols = set(spark.table(cfg["bronze"]).columns)
    for c in cfg["raw_cols"]:
        if c not in cols:
            log_drift(batch_id, f"silver/{cfg['name']}", cfg["bronze_short"], c, "MISSING_COLUMN", "string", "",
                      "FILLED_WITH_NULL")
    for c in sorted(cols - cfg["promoted"] - BRONZE_META):
        log_drift(batch_id, f"silver/{cfg['name']}", cfg["bronze_short"], c, "NEW_COLUMN_NOT_PROMOTED", "",
                  "string", "NOT_CARRIED_TO_SILVER (add it to the column map to promote it)")
    return cols


def read_bronze(cfg, batch_id, cols, chunk_value=None, use_chunk=False):
    b = spark.table(cfg["bronze"]).filter(F.col("batch_id") == batch_id)
    if use_chunk:
        b = b.filter(F.col(cfg["chunk"]).eqNullSafe(F.lit(chunk_value)))
    sel = [F.col(q(c)) if c in cols else F.lit(None).cast("string").alias(c) for c in cfg["raw_cols"] + cfg["extra_raw"]]
    return b.select(*sel, *[F.col(c) for c in LINEAGE_RAW])


def merge_upsert(src, cfg):
    view = "silver_src_" + uuid.uuid4().hex[:8]
    src.createOrReplaceTempView(view)
    names = [n for n, _ in cfg["spec"]]
    on = " AND ".join(f"t.{q(k)} = s.{q(k)}" for k in cfg["keys"])
    sets = ", ".join(f"{q(c)} = s.{q(c)}" for c in names if c not in cfg["keys"])
    ch = q(cfg["change_col"])
    result = spark.sql(f"""
        MERGE INTO {cfg['silver']} AS t USING {view} AS s ON {on}
        WHEN MATCHED AND s.collection_timestamp >= t.collection_timestamp AND NOT (t.{ch} <=> s.{ch})
             THEN UPDATE SET {sets}
        WHEN NOT MATCHED THEN INSERT ({', '.join(q(c) for c in names)}) VALUES ({', '.join('s.' + q(c) for c in names)})
        """).collect()[0].asDict()
    spark.catalog.dropTempView(view)
    return int(result.get("num_inserted_rows") or 0), int(result.get("num_updated_rows") or 0)


def process_chunk(cfg, batch_id, cols, chunk_value, use_chunk, c, cast_fail, val_nulls, reasons):
    raw = read_bronze(cfg, batch_id, cols, chunk_value, use_chunk)
    typed = cfg["build"](raw)
    no_issue = F.size("_issues") == 0
    aggs = [F.count(F.lit(1)).alias("n_read"), F.sum(F.when(~no_issue, 1).otherwise(0)).alias("n_bad")]
    aggs += [F.sum(F.when(F.col(cc).isNotNull() & caster(cc).isNull(), 1).otherwise(0)).alias(f"cf_{i}")
             for i, (label, cc, caster) in enumerate(cfg["casts"])]
    aggs += [F.sum(F.when(F.col(fl) & no_issue, 1).otherwise(0)).alias(f"vf_{i}") for i, fl in enumerate(cfg["flags"])]
    stats = typed.agg(*aggs).collect()[0].asDict()
    n_read, n_bad = int(stats["n_read"] or 0), int(stats["n_bad"] or 0)
    for i, (label, cc, caster) in enumerate(cfg["casts"]):
        cast_fail[label] = cast_fail.get(label, 0) + int(stats[f"cf_{i}"] or 0)
    for i, fl in enumerate(cfg["flags"]):
        val_nulls[fl[2:]] = val_nulls.get(fl[2:], 0) + int(stats[f"vf_{i}"] or 0)
    if n_read == 0:
        return
    bad, src, losers = prepare(cfg, typed)
    if n_bad:
        quarantine_df_to_table(quarantine_rows_bad(bad), cfg["silver_short"], batch_id)
        for r in quarantine_rows_bad(bad).groupBy("reason").count().collect():
            reasons[r["reason"]] = reasons.get(r["reason"], 0) + int(r["count"])
    n_dup = losers.count()
    if n_dup:
        quarantine_df_to_table(quarantine_rows_losers(losers, cfg["keys"]), cfg["silver_short"], batch_id)
        reasons["DUPLICATE_BUSINESS_KEY"] = reasons.get("DUPLICATE_BUSINESS_KEY", 0) + n_dup
    inserted, updated = merge_upsert(src, cfg)
    c.read += n_read
    c.quarantined += n_bad + n_dup
    c.inserted += inserted
    c.updated += updated
    c.skipped += n_read - n_bad - n_dup - inserted - updated


def run_table_batch(cfg, batch_id):
    """One Bronze table and one batch. A failure here is logged and never stops the other tables."""
    label = f"{cfg['bronze_short']}|batch_id={batch_id}"
    start, c = now_iso(), Counts()
    try:
        ensure_silver_columns(cfg, batch_id)
        cols = check_bronze_structure(cfg, batch_id)
        cast_fail, val_nulls, reasons = {}, {}, {}
        if cfg["chunk"]:
            values = [r[0] for r in spark.table(cfg["bronze"]).filter(F.col("batch_id") == batch_id)
                      .select(cfg["chunk"]).distinct().collect()]
            chunks = [(v, True) for v in values]
        else:
            chunks = [(None, False)]
        for value, use_chunk in chunks:
            process_chunk(cfg, batch_id, cols, value, use_chunk, c, cast_fail, val_nulls, reasons)
        cast_fail = {k: v for k, v in cast_fail.items() if v}
        val_nulls = {k: v for k, v in val_nulls.items() if v}
        for col_name, n in cast_fail.items():
            log_drift(batch_id, f"silver/{cfg['name']}", cfg["bronze_short"], col_name, "CAST_FAILURE", "typed",
                      "text", f"SET_NULL (rows: {n})", once=False)
        msg = json.dumps({"cast_failures": cast_fail, "validation_nulls": val_nulls, "quarantine_reasons": reasons})
        log_run(LAYER, batch_id, label, cfg["name"], cfg["silver_short"], start, "SUCCESS", c, msg)
        print(f"OK   {label}: read {c.read}, inserted {c.inserted}, updated {c.updated}, "
              f"skipped {c.skipped}, quarantined {c.quarantined}")
        return True, c
    except Exception as exc:
        log_error(cfg["name"], label, exc, type(exc).__name__, batch_id, traceback.format_exc())
        log_run(LAYER, batch_id, label, cfg["name"], cfg["silver_short"], start, "FAILURE", c, f"{type(exc).__name__}: {exc}")
        return False, c


def key_check(cfg):
    """Silver primary key check: duplicate and null keys per table, logged; the run fails on a problem."""
    start = now_iso()
    df = spark.table(cfg["silver"])
    total = df.count()
    dups = df.groupBy(*cfg["keys"]).count().filter("count > 1").count()
    nulls = df.filter(" OR ".join(f"{q(k)} IS NULL" for k in cfg["keys"])).count()
    ok = dups == 0 and nulls == 0
    c = Counts()
    c.read = total
    log_run("Silver data quality (primary key)", None, cfg["silver_short"], cfg["name"], cfg["silver_short"], start,
            "SUCCESS" if ok else "FAILURE", c,
            json.dumps({"key": cfg["keys"], "rows": total, "duplicate_keys": dups, "null_keys": nulls}))
    print(f"KEY CHECK {cfg['silver_short']}: rows={total}, duplicate keys={dups}, null keys={nulls}")
    return ok

### CELL 7 - Run: pick the batches, process them oldest first, one at a time, then the key check

In [ ]:
# CELL 7 - Run

def pending_batches(cfg):
    rows = (spark.table(cfg["bronze"]).groupBy("batch_id")
            .agg(F.min("collection_timestamp").alias("collected"),
                 F.date_format(F.min("collection_timestamp"), "yyyy-MM-dd").alias("day")).collect())
    out = []
    for r in rows:
        bid, day = r["batch_id"], r["day"]
        if P["batch_id"] and bid != P["batch_id"]:
            continue
        if FROM_B and bid[:len(FROM_B)] < FROM_B:
            continue
        if TO_B and bid[:len(TO_B)] > TO_B:
            continue
        if (START and day < START) or (END and day > END):
            continue
        out.append((r["collected"], bid))
    if not FORCE:
        done = {r["file_or_param"] for r in spark.table(T_RUNS)
                .filter((F.col("layer") == LAYER) & (F.col("status") == "SUCCESS")).select("file_or_param").collect()}
        out = [(c, b) for c, b in out if f"{cfg['bronze_short']}|batch_id={b}" not in done]
    return out


RUN_START, total, failed_any, changed = now_iso(), Counts(), False, False
if PARAM_ERRORS:
    log_run(f"{LAYER} (run)", None, "parameters", "all", "all", RUN_START, "FAILURE", total, "; ".join(PARAM_ERRORS))
    raise ValueError("Bad parameters: " + "; ".join(PARAM_ERRORS))

KNOWN_DRIFT.update(load_known_drift())
plan = {}
for _src in SOURCES:
    for key in pending_batches(CFG[_src]):
        plan.setdefault(key, []).append(CFG[_src])
order = sorted(plan)
print(f"Batches to process: {len(order)}")

for collected, bid in order:
    b_start, b_counts, b_ok = now_iso(), Counts(), True
    for cfg in plan[(collected, bid)]:
        ok, c = run_table_batch(cfg, bid)
        b_ok &= ok
        b_counts.add(c)
    changed |= (b_counts.inserted + b_counts.updated) > 0
    failed_any |= not b_ok
    total.add(b_counts)
    log_run(f"{LAYER} (batch)", bid, f"batch_id={bid}", "all", "all", b_start, "SUCCESS" if b_ok else "FAILURE",
            b_counts, f"tables: {[c['silver_short'] for c in plan[(collected, bid)]]}")

if changed or FORCE:
    for _src in SOURCES:
        failed_any |= not key_check(CFG[_src])

status = "FAILURE" if failed_any else "SUCCESS"
log_run(f"{LAYER} (run)", None, "run", P["source"], "all", RUN_START, status, total,
        f"batches processed: {len(order)}")
print(f"RUN {status}: batches={len(order)}, read={total.read}, inserted={total.inserted}, "
      f"updated={total.updated}, skipped={total.skipped}, quarantined={total.quarantined}")
if failed_any:
    raise Exception("Bronze-to-Silver finished with failures; see ops.error_logs and ops.pipeline_execution_logs")

### CELL 8 - Demo for the proof (schema change handled at the Silver step)

Passes a few sample rows through the **same** `arena_build` cleaning function that the real load uses. A row with `rating = 'n/a'` and a row with `rating = 'abc'` are quarantined, a row with a negative `vote_count` gets that value set to null, and the clean row passes. The quarantined rows appear in `ops.silver_quarantine_log` with `silver_table = 'demo'`. Nothing is written to the Silver tables.

In [ ]:
# CELL 8 - Demo for the proof (does not touch the Silver tables)

_demo_cols = ARENA_RAW_COLS + ["record_hash", "batch_id", "source_file", "raw_payload"]
_demo_schema = StructType([StructField(c, StringType(), True) for c in _demo_cols]
                          + [StructField("collection_timestamp", TimestampType(), True)])


def _row(tag, **kw):
    base = {c: None for c in _demo_cols}
    base.update(subset="text", category="overall", model_name="demo-model-" + tag, organization="demo-org",
                license="Proprietary", leaderboard_publish_date="2026-10-01", rating="1400.5", variance="40.0",
                rating_q975="1410.0", rating_q025="1390.0", vote_count="1000", rank="5", record_hash="demo_" + tag,
                batch_id="demo", source_file="demo.parquet", raw_payload=json.dumps({"tag": tag}))
    base.update(kw)
    return tuple(base[c] for c in _demo_cols) + (datetime.now(timezone.utc),)


_demo = spark.createDataFrame([
    _row("ok"),
    _row("rating_na", rating="n/a"),
    _row("rating_text", rating="abc"),
    _row("negative_votes", vote_count="-5"),
    _row("bad_date", leaderboard_publish_date="2026-13-45"),
], _demo_schema)

_typed = arena_build(_demo)
display(_typed.select("model_name", "rating", "vote_count", "leaderboard_publish_date", "_issues"))
_bad = _typed.filter(F.size("_issues") > 0)
_new = quarantine_df_to_table(quarantine_rows_bad(_bad), "demo", "demo")
print(f"Quarantined demo rows written (new): {_new}")
display(spark.sql(f"""SELECT silver_table, bronze_record_hash, column_name, bad_value, reason
                      FROM {T_SQ} WHERE silver_table = 'demo' ORDER BY bronze_record_hash"""))

### CELL 9 - Health check and proof queries

In [ ]:
# CELL 9 - Health check and proof queries

display(spark.sql(f"""
    SELECT 'silver.arena_leaderboard' AS table_name, COUNT(*) AS rows_loaded, MAX(load_timestamp) AS last_load FROM {T_S_ARENA}
    UNION ALL SELECT 'silver.epoch_models', COUNT(*), MAX(load_timestamp) FROM {T_S_EPOCH}"""))
display(spark.sql(f"""SELECT layer, file_or_param, status, rows_read, rows_inserted, rows_updated,
    rows_skipped_duplicate, rows_quarantined, start_time, end_time, message
    FROM {T_RUNS} WHERE layer LIKE 'Bronze-to-Silver%' OR layer LIKE 'Silver data quality%'
    ORDER BY load_timestamp DESC LIMIT 30"""))
display(spark.sql(f"""SELECT silver_table, reason, column_name, COUNT(*) AS records
    FROM {T_SQ} GROUP BY 1,2,3 ORDER BY 1,4 DESC"""))
display(spark.sql(f"""SELECT detected_at, dataset, column_name, event_type, action_taken
    FROM {T_DRIFT} WHERE dataset LIKE 'silver/%' ORDER BY detected_at DESC LIMIT 40"""))
display(spark.sql(f"""SELECT occurred_at, source, file_or_param, error_type, error_message
    FROM {T_ERRORS} WHERE layer = '{LAYER}' AND resolved = false ORDER BY occurred_at DESC LIMIT 20"""))

In [ ]:
# CELL 10 - Proof of the five rubric items (run after the first load; each query shows one item)

# Item 1: explicit schemas -> column names and types of the Silver tables
display(spark.sql(f"DESCRIBE TABLE {T_S_ARENA}"))
display(spark.sql(f"DESCRIBE TABLE {T_S_EPOCH}"))

# Item 2: MERGE re-run, no duplicates. Run this notebook twice with force_reload=true and compare:
#         the second run's table rows must show rows_inserted = 0 and rows_updated = 0.
display(spark.sql(f"""SELECT run_id, file_or_param, status, rows_read, rows_inserted, rows_updated,
    rows_skipped_duplicate, rows_quarantined FROM {T_RUNS} WHERE layer = '{LAYER}'
    ORDER BY start_time DESC LIMIT 12"""))
display(spark.sql(f"""SELECT file_or_param, status, message FROM {T_RUNS}
    WHERE layer = 'Silver data quality (primary key)' ORDER BY start_time DESC LIMIT 6"""))

# Item 3: backfill -> run with from_batch/to_batch (+ force_reload=true) and show only those batches here:
display(spark.sql(f"""SELECT batch_id, status, rows_read, rows_inserted, rows_updated, parameters_json
    FROM {T_RUNS} WHERE layer = '{LAYER} (batch)' ORDER BY start_time DESC LIMIT 10"""))